# Mix converted vocal + instrumental

Notebook này dùng để kiểm tra bước ghép lại sau pipeline:

```text
song -> UVR5 separate -> vocal + instrumental
vocal -> RVC convert -> converted_vocal
converted_vocal + instrumental -> final_song
```

Kỹ thuật dùng ở đây là audio mixing bằng FFmpeg `amix`, kèm `volume`, optional resample và limiter để giảm clipping.

## 1. Kiểm tra FFmpeg

Máy cần có `ffmpeg` và `ffprobe` trong PATH. RVC gốc cũng dùng FFmpeg để xử lý audio, nên đây là cách phù hợp nhất để test nhanh.

In [1]:
import json
import shlex
import subprocess
from pathlib import Path
from IPython.display import Audio, display


def run_cmd(cmd):
    print("$", " ".join(shlex.quote(str(part)) for part in cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")
    return proc.stdout.strip()


run_cmd(["ffmpeg", "-version"]).splitlines()[0]


$ ffmpeg -version


'ffmpeg version N-118034-gd21134313f-20241209 Copyright (c) 2000-2024 the FFmpeg developers'

## 2. Nhập file cần mix

Điền 2 đường dẫn local:

- `instrumental_path`: nhạc nền đã tách từ bài hát gốc
- `converted_vocal_path`: vocal đã được đổi giọng bằng RVC

Output sẽ ghi ra `mixed_output/final_mix.wav` mặc định.

In [2]:
# TODO: sửa 2 đường dẫn này theo file của bạn.
instrumental_path = Path(r"C:\Users\TEMP.THANHNGAN13.005\Downloads\separation_outputs_user_1779177387487_d34bd6a9-50f3-4150-8014-4b03f1edf898_sep_1780290935326_adb56d4c-1252-4013-8754-8fc38b47b2be_instrumental.wav")
converted_vocal_path = Path(r"C:\Users\TEMP.THANHNGAN13.005\Downloads\infer_outputs_user_1779177387487_d34bd6a9-50f3-4150-8014-4b03f1edf898_conv_1780291755683_ce0eb069-807c-4aeb-82a0-a60e485e3ca5.wav")

output_dir = Path("mixed_output")
output_path = output_dir / "final_mix.wav"

# Gain tuyến tính. 1.0 = giữ nguyên volume, 0.8 = giảm 20%.
instrumental_gain = 0.85
vocal_gain = 0.95

# 44100 hoặc 48000 đều được. None = giữ theo FFmpeg tự xử lý.
target_sample_rate = 44100

# Limiter giúp tránh rè/clipping sau khi cộng 2 waveform.
use_limiter = True


In [3]:
for label, path in {
    "instrumental": instrumental_path,
    "converted_vocal": converted_vocal_path,
}.items():
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path}")
    print(label, "=>", path.resolve())

output_dir.mkdir(parents=True, exist_ok=True)
print("output =>", output_path.resolve())


instrumental => C:\Users\TEMP.THANHNGAN13.005\Downloads\separation_outputs_user_1779177387487_d34bd6a9-50f3-4150-8014-4b03f1edf898_sep_1780290935326_adb56d4c-1252-4013-8754-8fc38b47b2be_instrumental.wav
converted_vocal => C:\Users\TEMP.THANHNGAN13.005\Downloads\infer_outputs_user_1779177387487_d34bd6a9-50f3-4150-8014-4b03f1edf898_conv_1780291755683_ce0eb069-807c-4aeb-82a0-a60e485e3ca5.wav
output => D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_my_server\mixed_output\final_mix.wav


## 3. Xem thông tin audio

Cell này dùng `ffprobe` để xem duration, sample rate và channel trước khi mix.

In [4]:
def probe_audio(path: Path) -> dict:
    raw = run_cmd([
        "ffprobe",
        "-v", "error",
        "-print_format", "json",
        "-show_streams",
        "-show_format",
        str(path),
    ])
    data = json.loads(raw)
    audio_stream = next(s for s in data["streams"] if s.get("codec_type") == "audio")
    return {
        "path": str(path),
        "duration": float(data.get("format", {}).get("duration", 0.0)),
        "sample_rate": int(audio_stream.get("sample_rate", 0)),
        "channels": int(audio_stream.get("channels", 0)),
        "codec": audio_stream.get("codec_name", ""),
    }


probe_audio(instrumental_path), probe_audio(converted_vocal_path)


$ ffprobe -v error -print_format json -show_streams -show_format 'C:\Users\TEMP.THANHNGAN13.005\Downloads\separation_outputs_user_1779177387487_d34bd6a9-50f3-4150-8014-4b03f1edf898_sep_1780290935326_adb56d4c-1252-4013-8754-8fc38b47b2be_instrumental.wav'
$ ffprobe -v error -print_format json -show_streams -show_format 'C:\Users\TEMP.THANHNGAN13.005\Downloads\infer_outputs_user_1779177387487_d34bd6a9-50f3-4150-8014-4b03f1edf898_conv_1780291755683_ce0eb069-807c-4aeb-82a0-a60e485e3ca5.wav'


({'path': 'C:\\Users\\TEMP.THANHNGAN13.005\\Downloads\\separation_outputs_user_1779177387487_d34bd6a9-50f3-4150-8014-4b03f1edf898_sep_1780290935326_adb56d4c-1252-4013-8754-8fc38b47b2be_instrumental.wav',
  'duration': 244.016327,
  'sample_rate': 44100,
  'channels': 2,
  'codec': 'pcm_s16le'},
 {'path': 'C:\\Users\\TEMP.THANHNGAN13.005\\Downloads\\infer_outputs_user_1779177387487_d34bd6a9-50f3-4150-8014-4b03f1edf898_conv_1780291755683_ce0eb069-807c-4aeb-82a0-a60e485e3ca5.wav',
  'duration': 243.98,
  'sample_rate': 40000,
  'channels': 1,
  'codec': 'pcm_s16le'})

## 4. Mix bằng FFmpeg

Công thức chính:

```text
final[t] = instrumental[t] * instrumental_gain + converted_vocal[t] * vocal_gain
```

`duration=longest` giữ output theo track dài hơn. Nếu một track ngắn hơn, FFmpeg sẽ tự pad silence.

In [5]:
def mix_vocal_with_instrumental(
    instrumental: Path,
    vocal: Path,
    output: Path,
    instrumental_volume: float = 0.85,
    vocal_volume: float = 0.95,
    sample_rate: int | None = 44100,
    limiter: bool = True,
):
    output.parent.mkdir(parents=True, exist_ok=True)

    # aformat ép cả 2 input về stereo float để mix ổn định dù input mono/stereo khác nhau.
    filter_parts = [
        f"[0:a]volume={instrumental_volume},aformat=sample_fmts=fltp:channel_layouts=stereo[a0]",
        f"[1:a]volume={vocal_volume},aformat=sample_fmts=fltp:channel_layouts=stereo[a1]",
        "[a0][a1]amix=inputs=2:duration=longest:dropout_transition=0:normalize=0[mix0]",
    ]

    current = "[mix0]"
    if limiter:
        filter_parts.append(f"{current}alimiter=limit=0.98[mix1]")
        current = "[mix1]"
    if sample_rate:
        filter_parts.append(f"{current}aresample={sample_rate}[out]")
        current = "[out]"

    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(instrumental),
        "-i", str(vocal),
        "-filter_complex", ";".join(filter_parts),
        "-map", current,
        "-c:a", "pcm_s16le",
        str(output),
    ]
    run_cmd(cmd)
    return output


mixed = mix_vocal_with_instrumental(
    instrumental=instrumental_path,
    vocal=converted_vocal_path,
    output=output_path,
    instrumental_volume=instrumental_gain,
    vocal_volume=vocal_gain,
    sample_rate=target_sample_rate,
    limiter=use_limiter,
)

print("Mixed file:", mixed.resolve())
probe_audio(mixed)


$ ffmpeg -y -i 'C:\Users\TEMP.THANHNGAN13.005\Downloads\separation_outputs_user_1779177387487_d34bd6a9-50f3-4150-8014-4b03f1edf898_sep_1780290935326_adb56d4c-1252-4013-8754-8fc38b47b2be_instrumental.wav' -i 'C:\Users\TEMP.THANHNGAN13.005\Downloads\infer_outputs_user_1779177387487_d34bd6a9-50f3-4150-8014-4b03f1edf898_conv_1780291755683_ce0eb069-807c-4aeb-82a0-a60e485e3ca5.wav' -filter_complex '[0:a]volume=0.85,aformat=sample_fmts=fltp:channel_layouts=stereo[a0];[1:a]volume=0.95,aformat=sample_fmts=fltp:channel_layouts=stereo[a1];[a0][a1]amix=inputs=2:duration=longest:dropout_transition=0:normalize=0[mix0];[mix0]alimiter=limit=0.98[mix1];[mix1]aresample=44100[out]' -map '[out]' -c:a pcm_s16le 'mixed_output\final_mix.wav'
Mixed file: D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_my_server\mixed_output\final_mix.wav
$ ffprobe -v error -print_format json -show_streams -show_format 'mixed_output\final_mix.wav'


{'path': 'mixed_output\\final_mix.wav',
 'duration': 244.016327,
 'sample_rate': 44100,
 'channels': 2,
 'codec': 'pcm_s16le'}

## 5. Nghe thử trong notebook

Nếu file dài, Jupyter có thể load hơi chậm. Khi đó mở file output trực tiếp bằng audio player ngoài.

In [ ]:
display(Audio(str(output_path)))


## 6. Tinh chỉnh nhanh

Nếu vocal bị nhỏ, tăng `vocal_gain` lên `1.05` hoặc `1.1`.

Nếu nhạc nền lấn vocal, giảm `instrumental_gain` xuống `0.75` hoặc `0.7`.

Nếu output bị rè, giảm cả hai gain hoặc bật `use_limiter=True`.

In [ ]:
# Ví dụ tạo bản vocal nổi hơn.
output_path_louder_vocal = output_dir / "final_mix_vocal_louder.wav"

mix_vocal_with_instrumental(
    instrumental=instrumental_path,
    vocal=converted_vocal_path,
    output=output_path_louder_vocal,
    instrumental_volume=0.75,
    vocal_volume=1.05,
    sample_rate=target_sample_rate,
    limiter=True,
)

display(Audio(str(output_path_louder_vocal)))


## 7. Xuất MP3 nếu cần

WAV nên dùng để kiểm chất lượng. Nếu cần file nhẹ để nghe thử hoặc gửi qua app, export MP3.

In [ ]:
mp3_output = output_dir / "final_mix.mp3"

run_cmd([
    "ffmpeg",
    "-y",
    "-i", str(output_path),
    "-codec:a", "libmp3lame",
    "-q:a", "2",
    str(mp3_output),
])

print("MP3:", mp3_output.resolve())


In [2]:
from pydub import AudioSegment

# 1. Đọc các file đã được tách từ thư mục đầu ra
vocals = AudioSegment.from_file(r"D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone\infer_output_full_explained.wav")
instrumental = AudioSegment.from_file(r"D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\project\audio_separator_application\python-audio-separator\output\song_(other)_vocals_mel_band_roformer.mp3")

# 2. Trộn hai tệp âm thanh đè lên nhau (overlay)
# Bạn có thể dùng tham số gain_during_overlay để chỉnh to nhỏ của nhạc cụ hoặc giọng hát
combined = vocals.overlay(instrumental)

# 3. Xuất kết quả ra tệp mới
combined.export("merged_song.mp3", format="mp3", bitrate="320k")
print("Ghép nhạc thành công!")


Ghép nhạc thành công!
